In [ ]:
import os
os.chdir('/home/linhang/workbench/workbench/Earthquake_predictor/dataset')
os.chdir('/home/linhang/workbench/workbench/Earthquake_predictor/')
from dataset.dataset_utils import *
from model.ES_net_mixer import *
from model import LightingModel
from torch.utils.data import DataLoader
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import json

In [ ]:
dataset = get_dataset(
    data_dir="/home/linhang/workbench/Earthquake_data/",
    window_size=140,
    forecast_horizon=14,
    lape_dim=30,
    geo_percentage=0.3,
    sem_percentage=0.3,
    time_resolution=14,
    earthquake_catalog_window=1400
    )

In [ ]:
def find_activate_area(earthquake_history,topk,earthquake_future):
    energy_catalog_sum = earthquake_future.sum(axis = 1).cpu().numpy()
    topk_index = np.argsort(energy_catalog_sum)[-topk:]
    earthquake_history_topk = earthquake_history[:,topk_index,:].squeeze(-1).cpu().numpy()
    earthquake_future_topk = earthquake_future[topk_index].permute(1,0).cpu().numpy()
    return earthquake_history_topk,earthquake_future_topk,topk_index

In [ ]:
model = LightingModel.load_from_checkpoint("/home/linhang/workbench/workbench/Earthquake_predictor/Result/ES_net_mixer_longterm_california/checkpoints/last.ckpt")
model.eval()

In [12]:
val_loader = DataLoader(
    dataset, batch_size=1, shuffle=True, collate_fn=dataset.collate_fn,num_workers=127
)

In [ ]:
model.predict_step

In [ ]:
for i in range(4):
    plt.plot(range(len(earthquake_history_topk[:,i])),earthquake_history_topk[:,i])
    plt.plot(range(len(earthquake_history_topk[:,i]),len(earthquake_history_topk[:,i])+len(earthquake_future_topk[:,i])),earthquake_future_topk[:,i])
    plt.show()